# 🔧 DOMIAN — Provisioning de Nodos con Swarm Bridge
## Genera instrucciones personalizadas para cada nodo ESP32-S3

Este notebook genera los comandos exactos de flash y provisioning para activar:
- ✅ Conexión WiFi
- ✅ Apuntar al servidor RuView
- ✅ Swarm Bridge para sincronización entre nodos
- ✅ CrossViewpointAttention (fusión multistática)

---
**Prerequisitos:**
```bash
pip install esptool esp-idf-nvs-partition-gen
```

## Paso 1 — Configuración general

In [ ]:
# ══════════════════════════════════════════════
# CONFIGURACIÓN — Edita estos valores
# ══════════════════════════════════════════════

WIFI_SSID     = "VTR-2889380"          # Tu red WiFi
WIFI_PASSWORD = ""                      # ← Ingresa tu contraseña aquí
SERVER_IP     = "192.168.0.8"          # IP del servidor RuView (Mac)
SERVER_PORT   = 5005                    # Puerto UDP
HTTP_PORT     = 8000                    # Puerto HTTP del servidor
NUM_NODOS     = 4                       # Cantidad de nodos (1-4)

# Ruta al repositorio
import os
REPO_DIR      = os.path.expanduser("~/Documents/RuView")
FIRMWARE_DIR  = f"{REPO_DIR}/firmware/esp32-csi-node"
BINS_DIR      = f"{FIRMWARE_DIR}/release_bins"
PROVISION_PY  = f"{FIRMWARE_DIR}/provision.py"

# ══════════════════════════════════════════════
# Canales por nodo (no cambiar salvo necesidad)
# ══════════════════════════════════════════════
CANALES = {1: 1, 2: 6, 3: 11, 4: 1}

# Validaciones
assert 1 <= NUM_NODOS <= 4, "NUM_NODOS debe ser entre 1 y 4"
assert WIFI_PASSWORD, "❌ Debes ingresar WIFI_PASSWORD"
assert os.path.exists(PROVISION_PY), f"❌ No encontrado: {PROVISION_PY}"
assert os.path.exists(BINS_DIR), f"❌ No encontrado: {BINS_DIR}"

SEED_URL = f"http://{SERVER_IP}:{HTTP_PORT}"

print(f"✅ Configuración válida")
print(f"   Red WiFi:    {WIFI_SSID}")
print(f"   Servidor:    {SERVER_IP}:{SERVER_PORT}")
print(f"   Seed URL:    {SEED_URL}")
print(f"   Nodos:       {NUM_NODOS}")
print(f"   Firmware:    {BINS_DIR}")
print(f"   Provision:   {PROVISION_PY}")

## Paso 2 — Detectar puertos disponibles

In [ ]:
import subprocess, glob

def detectar_puertos():
    """Detecta puertos seriales disponibles en Mac/Linux/Windows"""
    import sys
    if sys.platform == 'darwin':
        puertos = glob.glob('/dev/tty.usbmodem*') + glob.glob('/dev/tty.usbserial*')
    elif sys.platform == 'linux':
        puertos = glob.glob('/dev/ttyUSB*') + glob.glob('/dev/ttyACM*')
    else:  # Windows
        import serial.tools.list_ports
        puertos = [p.device for p in serial.tools.list_ports.comports()]
    return sorted(puertos)

puertos = detectar_puertos()
print(f"Puertos detectados: {len(puertos)}")
for i, p in enumerate(puertos):
    print(f"  [{i}] {p}")

if not puertos:
    print("\n⚠️  Sin puertos detectados — conecta un nodo por USB y vuelve a ejecutar")
else:
    print(f"\n✅ Conecta los nodos uno a uno para hacer el provisioning")

## Paso 3 — Generar instrucciones para cada nodo

In [ ]:
# Configuración de posiciones de nodos (metros)
POSICIONES = {
    1: (0.0,   0.0,   1.5),
    2: (0.0,   3.5,   1.5),
    3: (5.7,   3.5,   1.5),
    4: (5.7,   0.0,   1.5),
}

UBICACIONES = {
    1: "esquina inferior izquierda",
    2: "esquina superior izquierda",
    3: "esquina superior derecha",
    4: "esquina inferior derecha",
}

ROL = {1: "LÍDER (coordina el mesh)", 2: "FOLLOWER", 3: "FOLLOWER", 4: "FOLLOWER"}

print("=" * 65)
print(" DOMIAN — Instrucciones de Provisioning con Swarm Bridge")
print("=" * 65)
print()

for nodo_id in range(1, NUM_NODOS + 1):
    canal   = CANALES[nodo_id]
    pos     = POSICIONES[nodo_id]
    ubic    = UBICACIONES[nodo_id]
    rol     = ROL[nodo_id]

    print(f"{'━'*65}")
    print(f" NODO {nodo_id} — {rol}")
    print(f" Posición: {ubic} ({pos[0]}, {pos[1]}, {pos[2]})m")
    print(f" Canal WiFi: {canal}")
    print(f"{'━'*65}")
    print()

    print(" PASO A — Conecta el nodo por USB")
    print(f"   Luego detecta el puerto:")
    print(f"   ls /dev/tty.*      # Mac")
    print(f"   # Busca algo como: /dev/tty.usbmodemXXXXX")
    print()

    print(" PASO B — Modo bootloader")
    print("   1. Mantén presionado el botón BOOT")
    print("   2. Presiona y suelta RESET")
    print("   3. Suelta BOOT")
    print()

    print(" PASO C — Flash firmware (reemplaza PORT con tu puerto)")
    print(f"   cd {BINS_DIR}")
    print()
    flash_cmd = (
        f"   esptool --chip esp32s3 --port PORT --baud 460800 write_flash \\\'\n"
        f"     0x0 bootloader.bin \\\'\n"
        f"     0x8000 partition-table.bin \\\'\n"
        f"     0xd000 ota_data_initial.bin \\\'\n"
        f"     0x20000 esp32-csi-node.bin"
    )
    print(flash_cmd)
    print()

    print(" PASO D — Provisioning con Swarm Bridge")
    print(f"   cd {FIRMWARE_DIR}")
    print()
    prov_cmd = (
        f"   python3 provision.py \\\'\n"
        f"     --port PORT \\\'\n"
        f"     --ssid \"{WIFI_SSID}\" \\\'\n"
        f"     --password \"{WIFI_PASSWORD}\" \\\'\n"
        f"     --target-ip {SERVER_IP} \\\'\n"
        f"     --node-id {nodo_id} \\\'\n"
        f"     --channel {canal} \\\'\n"
        f"     --seed-url {SEED_URL}"
    )
    print(prov_cmd)
    print()

    print(" PASO E — Verificar")
    print(f"   curl http://{SERVER_IP}:{HTTP_PORT}/api/v1/nodes")
    print(f"   # Debe aparecer node_id={nodo_id} con status=active")
    print()

print("=" * 65)
print(" VERIFICACIÓN FINAL — después de provisionar todos los nodos")
print("=" * 65)
print()
print(" 1. Verifica nodos activos:")
print(f"    curl http://{SERVER_IP}:{HTTP_PORT}/api/v1/nodes")
print()
print(" 2. Verifica mesh sync:")
print(f"    curl http://{SERVER_IP}:{HTTP_PORT}/api/v1/mesh/metrics")
print(f"    # Objetivo: leader=1  follower={NUM_NODOS-1}  no_sync=0")
print()
print(" 3. Comando servidor con todas las posiciones:")
node_positions = ";".join(
    f"{POSICIONES[i][0]},{POSICIONES[i][1]},{POSICIONES[i][2]}"
    for i in range(1, NUM_NODOS + 1)
)
print(f"   cargo run -p wifi-densepose-sensing-server -- \\")
print(f"     --source esp32 --bind-addr 0.0.0.0 \\")
print(f"     --udp-port {SERVER_PORT} --http-port {HTTP_PORT} \\")
print(f"     --allowed-host {SERVER_IP} --allowed-host localhost \\")
print(f"     --node-positions \"{node_positions}\" \\")
print(f"     --load-rvf ~/Documents/RuView/_Domian/demo/models/laoracion.rvf")

## Paso 4 — Provisioning interactivo (un nodo a la vez)

In [ ]:
import subprocess, sys

def flash_nodo(puerto, nodo_id):
    """Flash firmware en el nodo"""
    print(f"\n🔥 Flasheando N{nodo_id} en {puerto}...")
    cmd = [
        "esptool", "--chip", "esp32s3",
        "--port", puerto,
        "--baud", "460800",
        "write_flash",
        "0x0",    f"{BINS_DIR}/bootloader.bin",
        "0x8000", f"{BINS_DIR}/partition-table.bin",
        "0xd000", f"{BINS_DIR}/ota_data_initial.bin",
        "0x20000",f"{BINS_DIR}/esp32-csi-node.bin"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if "Hash of data verified" in result.stdout:
        print(f"✅ Flash OK")
        return True
    else:
        print(f"❌ Error: {result.stderr[-200:]}")
        return False

def provision_nodo(puerto, nodo_id):
    """Provisioning con Swarm Bridge"""
    canal = CANALES[nodo_id]
    print(f"\n⚙️  Provisionando N{nodo_id} (canal {canal}, seed_url={SEED_URL})...")
    cmd = [
        sys.executable, PROVISION_PY,
        "--port",      puerto,
        "--ssid",      WIFI_SSID,
        "--password",  WIFI_PASSWORD,
        "--target-ip", SERVER_IP,
        "--node-id",   str(nodo_id),
        "--channel",   str(canal),
        "--seed-url",  SEED_URL,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if "NVS provisioning complete" in result.stdout:
        print(f"✅ Provisioning OK")
        return True
    else:
        print(f"⚠️  Output: {result.stdout[-300:]}")
        print(f"⚠️  Error:  {result.stderr[-200:]}")
        return False

# Ejecutar provisioning para cada nodo
resultados = {}

for nodo_id in range(1, NUM_NODOS + 1):
    print()
    print(f"{'━'*50}")
    print(f" NODO {nodo_id} — {ROL[nodo_id]}")
    print(f"{'━'*50}")
    print()
    print(" Conecta el nodo por USB y pon en modo bootloader:")
    print(" 1. Mantén BOOT → 2. Presiona RESET → 3. Suelta BOOT")
    print()

    puerto = input(f" Puerto serial para N{nodo_id} (ej: /dev/tty.usbmodemXXXX): ").strip()

    if not puerto:
        print(f"  ⚠️  Saltando N{nodo_id}")
        resultados[nodo_id] = 'saltado'
        continue

    # Flash
    flash_ok = flash_nodo(puerto, nodo_id)
    if not flash_ok:
        resp = input("  ¿Continuar con provisioning de todas formas? (s/n): ")
        if resp.lower() != 's':
            resultados[nodo_id] = 'error_flash'
            continue

    # Provision
    prov_ok = provision_nodo(puerto, nodo_id)
    resultados[nodo_id] = 'ok' if prov_ok else 'error_provision'

    print(f"\n✅ N{nodo_id} completado. Desconecta el USB y conecta el siguiente nodo.")

# Resumen
print()
print("=" * 50)
print(" RESUMEN")
print("=" * 50)
for nodo_id, estado in resultados.items():
    icono = "✅" if estado == 'ok' else "⚠️ " if estado == 'saltado' else "❌"
    print(f"  N{nodo_id}: {icono} {estado}")

## Paso 5 — Verificar estado del mesh

In [ ]:
import urllib.request, json, time

print("Esperando 15 segundos para que los nodos reconecten...")
time.sleep(15)

def fetch(path):
    try:
        with urllib.request.urlopen(f"http://{SERVER_IP}:{HTTP_PORT}{path}", timeout=5) as r:
            return json.loads(r.read())
    except Exception as e:
        return {"error": str(e)}

# Verificar nodos
nodes = fetch("/api/v1/nodes")
print(f"\nNodos conectados: {nodes.get('total', 0)}/{NUM_NODOS}")
for n in nodes.get("nodes", []):
    status = "✅" if n['status'] == 'active' else "⚠️"
    print(f"  {status} N{n['node_id']} — {n['motion_level']} — last_seen: {n['last_seen_ms']}ms")

# Verificar mesh
print()
mesh = fetch("/api/v1/mesh")
if "error" not in mesh:
    total_mesh = mesh.get("total", 0)
    print(f"Mesh nodes: {total_mesh}")
    for node_id, info in mesh.get("nodes", {}).items():
        role = info.get("role", "unknown")
        sync = info.get("is_valid", False)
        print(f"  N{node_id}: role={role} synced={sync}")

# Sensing actual
sensing = fetch("/api/v1/sensing/latest")
if "error" not in sensing:
    clf = sensing.get("classification", {})
    print(f"\nEstado actual:")
    print(f"  Motion:           {clf.get('motion_level')}")
    print(f"  Confidence:       {clf.get('confidence', 0):.3f}")
    print(f"  Estimated persons:{sensing.get('estimated_persons')}")

## Paso 6 — Comando final del servidor

Copia y ejecuta este comando en la terminal de la Mac para arrancar el servidor con todos los parámetros correctos.

In [ ]:
node_positions = ";".join(
    f"{POSICIONES[i][0]},{POSICIONES[i][1]},{POSICIONES[i][2]}"
    for i in range(1, NUM_NODOS + 1)
)

cmd = f"""cd ~/Documents/RuView/v2
cargo run -p wifi-densepose-sensing-server -- \\
  --source esp32 \\
  --bind-addr 0.0.0.0 \\
  --udp-port {SERVER_PORT} \\
  --http-port {HTTP_PORT} \\
  --allowed-host {SERVER_IP} \\
  --allowed-host localhost \\
  --node-positions "{node_positions}" \\
  --load-rvf ~/Documents/RuView/_Domian/demo/models/laoracion.rvf"""

print("Comando para arrancar el servidor:")
print()
print(cmd)
print()
print("Después de arrancar verifica:")
print(f"  curl http://{SERVER_IP}:{HTTP_PORT}/api/v1/mesh/metrics")
print(f"  # Objetivo: leader=1  follower={NUM_NODOS-1}  no_sync=0")